# W5Act1 — 01 Data Understanding

**Stage:** Data acquisition → Data understanding → Data validation

Everything in this notebook that runs automatically is *mechanical*: shape, dtypes,
missingness, duplicates, outlier flags. It is safe on any tabular dataset and tells you
nothing about what the data means.

The `## DECISION` cells are where the analysis actually starts. They are blank on purpose.

> **This notebook is a stage template, not a mandatory step.** Use the notebooks your
> question needs and delete the rest — a descriptive project may never open
> `04_modeling.ipynb`, and a messy dataset may spend most of its life in `01`.
> See `CHECKLIST.md` for the full workflow.

In [1]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Point this at your file in data/raw/. Everything below follows from it.
DATASET = 'iris_clean.csv'

## DECISION 1 — What question is this data supposed to answer?

Answer before you look at a single chart. A dataset explored without a question produces
a tour, not an analysis.

- **Question:** Classify an iris flower's species (Iris-setosa / Iris-versicolor /
  Iris-virginica) from its four sepal and petal measurements, using a support vector
  machine (SVM).
- **Who needs the answer:** This is a coursework exercise (MSE803) — the reader is the
  assessment, and the point is to demonstrate a correctly-built multivariate
  classification pipeline end to end, not to serve a downstream business user.
- **What would change your mind:** If held-out (test-set) accuracy did not clear the
  majority-class baseline by a meaningful margin, that would say either the SVM is
  misconfigured or the four measurements do not separate species well enough — a finding
  to report plainly, not a failure to hide.
- **"Good enough":** held-out accuracy meaningfully above the majority-class baseline
  (~33%, classes are close to balanced) — not training accuracy.

In [2]:
from src.data.loading import list_raw, load_raw

print('files in data/raw/:', list_raw())
df = load_raw(DATASET)
df.head()

files in data/raw/: ['.DS_Store', 'iris_clean.csv']


,sepal length,sepal width,petal length,petal width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Reading

One row is a single iris flower: four measurements in centimetres (sepal length, sepal
width, petal length, petal width) plus its recorded species. This is Fisher's classic
1936 iris dataset — a fixed historical sample of irises from the Gaspé Peninsula, not an
ongoing collection, so there is no time dimension to track. 150 rows, 50 of each of the
three species, as originally published.

In [3]:
from src.data.quality import profile, schema_report

profile(df)
schema_report(df)

rows: 150   columns: 5
memory: 13.9 KB

dtypes:
float64    4
str        1


,column,dtype,non_null,missing,missing_pct,unique,example
0,sepal length,float64,150,0,0.0,35,5.1
1,sepal width,float64,150,0,0.0,23,3.5
2,petal length,float64,150,0,0.0,43,1.4
3,petal width,float64,150,0,0.0,22,0.2
4,class,str,150,0,0.0,3,Iris-setosa


### Reading

All four measurement columns are already numeric (`float64`); `class` is text with
exactly 3 levels, as expected — nothing needs re-typing. Units are centimetres,
consistent with the standard iris dataset documentation. No surprises here.

In [4]:
from src.data.quality import missing_report
from src.visualization.plots import missingness_matrix

missing = missing_report(df)
display(missing)

if df.isna().any().any():
    fig, _ = missingness_matrix(df)
    save_fig(fig, '01_missingness')
    plt.show()
else:
    print('No missing values — skip the missingness DECISION below.')

complete rows: 150 of 150 (100.0%)
no missing values


,missing,missing_pct


No missing values — skip the missingness DECISION below.


## DECISION 2 — Why is the data missing, and does that permit imputation?

Counting gaps is mechanical; explaining them is not, and it decides whether filling them
is legitimate at all.

| Mechanism | Meaning | Imputation |
|---|---|---|
| MCAR | Missing at random, unrelated to anything | Safe; you lose precision, not validity |
| MAR | Missingness explained by *other observed* columns | Defensible if you model it on those columns |
| MNAR | Missingness depends on the missing value itself (high earners skip the income box) | Imputing here manufactures a convenient answer |

- **Mechanism, and the evidence for it:** N/A — `missing_report` above found 0 missing
  values in any column (150/150 rows complete). There is nothing to explain or fill.
- **Decision:** No imputation needed.

Nothing downstream should fill these values until this cell is answered.

In [5]:
from src.data.quality import cardinality_report, duplicate_report

# subset= names the columns that identify a record. Pass it once you know what a
# duplicate means here — it is a fact about your data, not something to infer.
duplicate_report(df, subset=None)
cardinality_report(df)

exact duplicate rows: 3


,column,unique,unique_ratio,note
2,petal length,43,0.287,
0,sepal length,35,0.233,
1,sepal width,23,0.153,
3,petal width,22,0.147,
4,class,3,0.020,


### Reading

`duplicate_report` found 3 exact duplicate rows: 2 extra copies of one Iris-setosa row
(4.9, 3.1, 1.5, 0.1) and 1 extra copy of one Iris-virginica row (5.8, 2.7, 5.1, 1.9). An
exact match across all four measurements *and* the label, at this precision, is far more
consistent with an export/copy artifact than three or two separate flowers happening to
measure identically — real biological replicates that precise would be a coincidence.
**Decision: drop them.** This is carried out in `02_eda.ipynb`'s cleaning cell via
`drop_exact_duplicates`, leaving 147 rows (48 setosa, 50 versicolor, 49 virginica).

`cardinality_report` shows no constant column (no column with a single value) and no
near-unique identifier column — `class` is the only categorical, with exactly the
expected 3 levels.

In [6]:
from src.data.quality import outlier_report

outliers = outlier_report(df)
outliers

Values flagged. Flagged is not the same as wrong -- decide and record why.


,column,n,iqr_flagged,z_flagged,lower_fence,upper_fence,min,max
0,sepal length,150,0,0,3.15,8.35,4.3,7.9
1,sepal width,150,4,1,2.05,4.05,2.0,4.4
2,petal length,150,0,0,-3.65,10.35,1.0,6.9
3,petal width,150,0,0,-1.95,4.05,0.1,2.5


## DECISION 3 — Are the flagged values errors or real observations?

`outlier_report` flags and never removes. A flagged value can be a typo, a unit mix-up, or
the single most interesting row in the dataset.

- **Values examined:** 4 sepal-width values flagged, all in `sepal width` — three wide
  Iris-setosa readings (4.4, 4.1, 4.2 cm) and one narrow Iris-versicolor reading (2.0 cm).
  All four sit just outside the 1.5×IQR fence (2.05–4.05 cm); only the 4.4 cm value also
  clears the z > 3 threshold.
- **Decision:** keep all of them. None are biologically impossible — sepal widths in this
  range are plausible for these species — and with n=150 (147 after dedup), a handful of
  tail values is the expected shape of a small dataset, not evidence of a measurement
  error.
- **Effect if you are wrong:** if any of these were really data-entry errors, they add a
  small amount of noise to the `sepal_width` feature, which could slightly widen the
  region an SVM learns as the class boundary near those points. Given petal measurements
  separate the species far more cleanly than sepal width does, this is unlikely to change
  which species gets predicted.

Excluding a row is a claim that it does not belong to the population you are studying.
Write down the claim.

In [7]:
from src.data.quality import quality_report
from src.visualization.presentation import save_table

summary = quality_report(df)
save_table(summary, '01_data_quality')

PROFILE
rows: 150   columns: 5
memory: 13.9 KB

dtypes:
float64    4
str        1

MISSING
complete rows: 150 of 150 (100.0%)
no missing values

DUPLICATES
exact duplicate rows: 3

OUTLIERS (flagged, not removed)
Values flagged. Flagged is not the same as wrong -- decide and record why.
      column   n  iqr_flagged  z_flagged  lower_fence  upper_fence  min  max
sepal length 150            0          0         3.15         8.35  4.3  7.9
 sepal width 150            4          1         2.05         4.05  2.0  4.4
petal length 150            0          0        -3.65        10.35  1.0  6.9
 petal width 150            0          0        -1.95         4.05  0.1  2.5

SUMMARY (one row per column)
      column   dtype  non_null  missing  missing_pct  unique     example  unique_ratio note  iqr_flagged  z_flagged
sepal length float64       150        0          0.0      35         5.1         0.233               0.0        0.0
 sepal width float64       150        0          0.0      23     

PosixPath('/Users/ericsmac/Documents/GitHub/MSE803-DA/W5Act1/outputs/tables/01_data_quality.csv')

## What this data can and cannot answer

Fill this in before moving on. It is the most useful cell in the notebook and the easiest
to skip.

**Can answer:**
- Can an iris flower's species be classified from its sepal and petal measurements? Yes —
  all four predictors (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`) and
  the label (`class`) are present for every row, with no missingness, so this is directly
  answerable with a supervised classifier (SVM, per DECISION 1).

**Cannot answer:**
- Anything about *why* species differ morphologically — this is measurement-plus-label
  data with no experimental design, so it supports classification, not causal
  explanation.
- Anything about drift or new time periods — this is a fixed historical sample (Fisher,
  1936) with no time dimension or repeated collection.
- How well a model generalises beyond n=147 and these three species — there is no
  external validation set from a different source or year.

**Next stage:** `02_eda` — clean the 3 duplicate rows found above, then proceed toward
`04_modeling` for the SVM. Description alone cannot answer DECISION 1's classification
question, and the dataset is small enough that a quick EDA pass is worthwhile before
splitting the data.